# Submitting a Recipe Fine-Tuning Job - HyperPod CLI End-to-End Walkthrough

This example shows how to fine-tune a model using the **HyperPod CLI recipe job** experience (`hyp-recipe-job`). Recipes are pre-built fine-tuning configurations published to SageMaker JumpStart Hub — they include a Kubernetes job template and parameter spec, so you don't need to write any YAML.

The workflow is:
1. **`hyp init`** — fetch the recipe from SageMaker Hub and scaffold your job directory
2. **`hyp configure`** — set your training parameters
3. **`hyp validate`** — verify the configuration is complete and valid
4. **`hyp create`** — render and submit the Kubernetes job

This example assumes you have completed the **Setup instructions** in [00-getting-started/00-setup.md](../00-getting-started/00-setup.md) and have a HyperPod EKS cluster with your kubeconfig configured.

## Prerequisites

- HyperPod EKS cluster with kubeconfig configured
- `sagemaker-hyperpod` CLI installed (`pip install sagemaker-hyperpod`)
- FSx for Lustre volume mounted at `/data` with training data available
- AWS credentials with `sagemaker:DescribeHubContent`, `sagemaker:ListHubContents`, and `s3:GetObject` permissions

## Set Cluster Context

Configure to point at your HyperPod EKS cluster before running any other commands.

In [ ]:
CLUSTER_NAME = "<your-cluster-name>"  # Replace with your HyperPod cluster name
!hyp set-cluster-context --cluster-name {CLUSTER_NAME}

## Step 0: Configuration

Set your job name and working directory.

In [ ]:
import os

JOB_NAME = "qwen3-sft-recipe-job"   # Change to a unique name
JOB_DIR  = f"./{JOB_NAME}"          # Local directory for job files
NAMESPACE = "default"

os.makedirs(JOB_DIR, exist_ok=True)
print(f"Job directory: {os.path.abspath(JOB_DIR)}")

## Step 1: Initialize the Recipe Job

The `hyp init hyp-recipe-job` command fetches the recipe from SageMaker JumpStart Hub and creates three files in your job directory:
- `config.yaml` — your editable training parameters
- `.override_spec.json` — the parameter schema (used by `configure` and `validate`)
- `k8s.jinja` — the Kubernetes job template

### Specifying your model

You can identify the model using either a **JumpStart model ID** or a **HuggingFace model ID**:

```bash
# Option A: JumpStart model ID
hyp init hyp-recipe-job --model-id huggingface-reasoning-qwen3-06b --technique SFT --instance-type ml.g5.48xlarge

# Option B: HuggingFace model ID (resolved automatically via JumpStart Hub search)
hyp init hyp-recipe-job --huggingface-model-id Qwen/Qwen3-0.6B --technique SFT --instance-type ml.g5.48xlarge
```

### Instance type selection

If you omit `--instance-type`, the CLI will **automatically query your HyperPod clusters** and find clusters with instance types supported by the recipe and technique you selected. You will be presented with a list of compatible clusters to choose from.

Supported job types:\n- **Fine-tuning**: `SFT`, `DPO`, `CPT`, `PPO`, `RLAIF`, `RLVR`\n- **Evaluation**: `deterministic`, `LLMAJ`

In [ ]:
# Option A: JumpStart model ID
!hyp init hyp-recipe-job {JOB_DIR} \
    --model-id huggingface-reasoning-qwen3-06b \
    --technique SFT \
    --instance-type ml.g5.48xlarge

In [ ]:
# Option B: HuggingFace model ID (uncomment to use)
# !hyp init hyp-recipe-job {JOB_DIR} \
#     --huggingface-model-id Qwen/Qwen3-0.6B \
#     --technique SFT \
#     --instance-type ml.g5.48xlarge

> **Note — Option C requires a terminal.** Omitting `--instance-type` triggers an interactive cluster selection prompt, which is not supported in Jupyter notebooks. Run this command in a terminal instead:
>
> ```bash
> hyp init hyp-recipe-job ./<job-dir> \
>     --huggingface-model-id Qwen/Qwen3-0.6B \
>     --technique SFT
> ```

In [ ]:
# Verify files were created
!ls -la {JOB_DIR}

## Step 2: Configure Training Parameters

Use `hyp configure` to set your training parameters. Run this from inside the job directory.

You can see all available parameters with `hyp configure --help` (run from inside the job directory).

In [ ]:
%%bash -s "$JOB_NAME" "$NAMESPACE"
cd $1
hyp configure \
    --name $1 \
    --namespace $2 \
    --data-path /data/recipes-data/sft/zc_train_256.jsonl \
    --global-batch-size 8 \
    --learning-rate 0.0001 \
    --lr-warmup-ratio 0.1 \
    --max-epochs 5 \
    --output-path /data/output/qwen3-sft \
    --results-directory /data/results/qwen3-sft \
    --resume-from-path /data/output/qwen3-sft \
    --training-data-name zc_train_256 \
    --validation-data-name zc_train_256 \
    --validation-data-path /data/recipes-data/sft/zc_train_256.jsonl \
    --train-val-split-ratio 0.9 \
    --instance-type ml.g5.48xlarge

In [ ]:
# Review the generated config
!cat {JOB_DIR}/config.yaml

## Step 3: Validate Configuration

Validate that all required fields are set and values are within allowed ranges.

In [ ]:
%%bash -s "$JOB_NAME"
cd $1
hyp validate

## Step 4: Submit the Job

`hyp create` renders the Kubernetes YAML from `k8s.jinja` + `config.yaml` and submits it to your cluster. The rendered files are saved under `run/<timestamp>/` for reference.

In [ ]:
%%bash -s "$JOB_NAME"
cd $1
hyp create

## Step 5: Monitor the Job

In [ ]:
# Check job status
!kubectl get hyperpodpytorchjob {JOB_NAME} -n {NAMESPACE}

In [ ]:
# List pods
!hyp list-pods hyp-recipe-job --job-name {JOB_NAME} --namespace {NAMESPACE}

In [ ]:
# Stream logs from the first pod (replace pod name from list-pods output above)
# !kubectl logs -f <pod-name> -n {NAMESPACE}

## Step 6: Clean Up

In [ ]:
!hyp delete hyp-recipe-job --job-name {JOB_NAME} --namespace {NAMESPACE}